# 38장 · 재정렬과 다중 질의 검색

재정렬은 기존 후보들의 순서를 바꾸고, 다중 질의 검색은 질문 표현을 늘려 후보를 모읍니다. 첫 검색에 정답 문서가 없는 경우 두 방법의 차이가 드러납니다.

대출 기간을 묻는 질문에서 마침표만 담은 청크가 1위가 됐습니다. 기존 후보를 다시 정렬하는 방법과 질의를 확장해 새 후보를 가져오는 방법을 같은 자료에서 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## sorted()를 이용한 후보 재정렬

37장의 16글자 청크 중 library-1:0에는 14일이 있고 library-1:1에는 마침표만 있습니다. 순위 차이를 눈으로 보기 위해 이 두 입력의 순서를 뒤집어 둡니다. 여기서 쓰는 점수는 수업용 단어 일치이며 학습된 재순위 모델의 결과와 구분합니다.

### 같은 후보 두 건에서 1위가 바뀌는 과정

**첫 후보에는 마침표만 있습니다.**　대출 기간을 담은 청크는 이미 둘째 자리에 있습니다.

```
1. library-1:1  .
2. library-1:0  도서 대출 기간은 14일입니다
```

**본문에 ‘대출’이 있는지 봅니다.**　첫 청크의 비교값은 False, 둘째 청크는 True입니다. 노트북의 정렬 규칙은 True를 앞으로 보냅니다.

```
library-1:1  False
library-1:0  True
```

**같은 두 후보의 순서가 달라집니다.**　맨 앞에서 본문을 하나만 읽어도 이제 14일이라는 내용을 확인할 수 있습니다.

```
1. library-1:0  도서 대출 기간은 14일입니다
2. library-1:1  .
```

**후보 집합과 재정렬 순위**　재정렬 전 첫 후보는 마침표 하나입니다. 대출 문장을 앞으로 보내도 전체 후보 수는 같습니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter38/index.html#practice-space)


### 셀 01 · 재정렬 전 후보 순위

후보 두 건을 그대로 받아 첫 번째 청크의 본문을 읽습니다.


In [1]:
candidates = [{'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1},
 {'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1}]
print(candidates[0]['id'], repr(candidates[0]['text']))

library-1:1 '.'


마침표만 있는 library-1:1이 보입니다. 원문은 맞지만 이 조각만 받은 답변 함수에는 기간을 확인할 숫자가 없습니다.


### 검색 후보와 재정렬 결과

<p>candidates에는 같은 원문에서 잘라 낸 청크 두 개가 있습니다. 첫 청크를 바로 고를 때와 두 후보의 순서를 다시 매긴 뒤 첫 청크를 고를 때, 답변에 쓸 수 있는 문장이 달라집니다.</p>


### 후보에 있어야 순위가 바뀝니다

<p>첫 후보의 본문은 마침표 하나이고, 대출 기간을 담은 문장은 둘째 후보입니다. 둘째 후보를 목록에서 빼고 정렬하면 마침표만 남습니다. 정렬 함수는 목록 밖의 문서를 새로 가져오지 않습니다.</p>


### 셀 02 · sorted()의 key와 reverse

같은 후보 안에서 ‘대출’이 있는지 채점해 가장 앞에 놓습니다.


In [2]:
ranked = sorted(candidates, key=lambda doc: '대출' in doc['text'], reverse=True)
print(ranked[0]['id'], ranked[0]['text'])

library-1:0 도서 대출 기간은 14일입니다


‘대출’을 포함하는 library-1:0이 2위에서 1위로 올라왔습니다.

재정렬의 효과를 확인하려면 후보 목록에서 library-1:0을 잠시 빼 보세요. 남은 마침표를 아무리 정렬해도 ‘14일’은 돌아오지 않습니다. 이 셀의 점수는 ‘대출’이 들어 있는지를 나타내는 불리언 값입니다. 학습된 재정렬 모델이 질문과 문장의 뜻을 얼마나 잘 비교하는지는 별도 실험에서 측정해야 합니다.

<div class="example-table"><table><caption>같은 후보 두 건의 재정렬</caption><thead><tr><th scope="col">청크 ID</th><th scope="col">청크 본문</th><th scope="col">검색 순위</th><th scope="col">키워드 정렬 후</th></tr></thead><tbody><tr><td>library-1:1</td><td>.</td><td>1</td><td>2</td></tr><tr><td>library-1:0</td><td>도서 대출 기간은 14일입니다</td><td>2</td><td>1</td></tr></tbody></table></div>


### 불리언 값으로 재정렬하기

<p>key 함수는 본문에 ‘대출’이 있으면 True를 반환합니다. reverse=True로 정렬하므로 대출 문장이 마침표보다 앞으로 옵니다. 이 작은 규칙은 질문과 후보를 함께 읽는 재정렬 모델이 맡을 일을 눈으로 확인하기 위한 예입니다.</p>


## 질의 확장과 청크 ID 중복 제거

‘책을 빌릴 수 있는 기간은?’을 세 낱말 벡터로 바꾸면 대출 항목이 0입니다. 이번에는 수업용 사전에 정한 표현 ‘대출 기간’을 더해 검색합니다. 원래 질문도 남겨 두고, 공백만 다른 표현은 한 번만 검색합니다.


### 과제 03 · 중복 질의 제거

원래 질문과 확장문을 순서대로 읽어 공백을 정리하고 중복 없이 남기세요.


In [3]:
queries = ['책을 빌릴 수 있는 기간은?', '대출 기간', '대출   기간']
# 첫 등장 순서를 지키며 중복을 제거하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>queries = [&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;, &#x27;대출   기간&#x27;]
unique = []
for query in queries:
    clean = &#x27; &#x27;.join(query.split())
    if clean not in unique:
        unique.append(clean)
print(unique)</code></pre>
<p>실행 결과</p><pre><code>[&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;]</code></pre>
<p>두 검색문이 남습니다. 검색문 중복을 줄였어도 양쪽 검색 결과에 같은 청크가 나올 수 있습니다. 결과를 합칠 때는 청크 ID 기준으로 한 번 더 확인합니다.</p>
</details>


### 원래 질문과 검색용 표현

<p>‘책을 빌릴 수 있는 기간은?’과 ‘대출 기간’은 같은 자료를 찾는 데 쓸 수 있는 표현입니다. 노트북에서는 검색용 문구를 직접 적습니다. 표현을 늘리면 서로 다른 검색 결과를 얻을 기회가 생기지만, 같은 문서를 되풀이해 찾을 수도 있습니다.</p>

<h3>두 질의가 찾은 청크를 한 목록으로 모으기</h3>

<p>셀 04의 두 검색어 ‘대출 기간은 며칠인가요?’와 ‘대출 기간’은 모두 library-1:0을 찾습니다. batches에는 한 건씩 담긴 목록 두 개가 생깁니다. 이 목록들을 펼친 뒤 dict.fromkeys로 같은 ID를 합치면 merged에는 library-1:0 한 건이 남습니다.</p>


### 질의와 청크의 중복 제거

<p>대출 기간과 대출   기간은 이 실습의 공백 정리 규칙을 거치면 같은 문구입니다. 두 번 검색해도 새 근거가 생기는 것은 아니므로 중복 호출을 줄입니다. 단어 순서가 다른 표현을 자동으로 같은 뜻이라고 판정하지는 않습니다.</p>

<p>검색 결과는 문장 내용이 같다는 이유로 합치지 않습니다. 서로 다른 원문 페이지가 같은 문장을 포함할 수 있기 때문입니다. 청크 ID를 기준으로 합치되 같은 ID의 본문·출처·페이지가 다르면 자료 충돌로 처리합니다. 원문 ID가 같은 서로 다른 청크를 하나로 합치면 필요한 문장을 잃을 수 있습니다.</p>


### 셀 04 · 검색 결과의 청크 ID 합집합

같은 고정 인덱스에서 검색문별 결과를 받아 첫 등장 순서대로 합칩니다. 이 예의 검색은 ‘대출’ 포함 여부로 찾는 작은 로컬 검색입니다.

과제 03에서는 ‘책을 빌릴 수 있는 기간은?’을 출발점으로 표현을 늘렸습니다. 이번 셀은 두 표현 모두 ‘대출’을 포함하는 별도 예입니다. 여기서 관찰할 것은 새 문서 발견이 아니라 두 검색이 돌려준 같은 ID를 한 번만 남기는 과정입니다.


In [4]:
index = [{'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1}]
search_queries = ['대출 기간은 며칠인가요?', '대출 기간']

def find_chunks(query):
    return [row['id'] for row in index if '대출' in query and '대출' in row['text']]

batches = [find_chunks(query) for query in search_queries]
merged = list(dict.fromkeys(chunk for batch in batches for chunk in batch))
print(batches)
print(merged)

[['library-1:0'], ['library-1:0']]
['library-1:0']


질문 표현 두 개가 같은 library-1:0을 찾았습니다. 검색은 두 번 했지만 답의 근거에는 청크 하나만 넘기면 됩니다. 원래 후보에 없던 청크가 추가되려면 확장한 검색문에서 실제로 다른 결과를 얻어야 합니다.


### 질의를 늘려도 같은 문서를 만날 수 있습니다

<p>두 검색의 반환 목록을 이어 붙이면 library-1:0이 두 번 나옵니다. dict.fromkeys로 첫 등장 순서를 유지하며 한 번만 남긴 것이 merged입니다. 서로 다른 근거가 늘었는지는 중복을 제거한 다음에 확인합니다.</p>


### 검색 횟수와 근거 수

<p>search_queries는 두 문장이고 find_chunks도 두 번 실행됩니다. 두 결과가 같은 청크 ID라서 merged에는 하나만 남습니다. 호출 수를 줄이는 것과 마지막에 중복 근거를 없애는 것은 서로 다른 작업입니다.</p>


## 돌아보기

키워드 재순위가 성공한 위 사례는 의도적으로 순서를 뒤집은 연습입니다. 실제 채택 여부는 다음 장의 같은 평가 질문에서 나온 결과와 추가 작업 횟수로 판단하세요.

### 확장부터 최종 문서까지

**원질의와 확장 목록**　고정 규칙이 문구를 반환합니다. 실제 모델 생성이 아닙니다.

```
과제 03 · 책을 빌릴 수 있는 기간은?
확장: 대출 기간 / 대출   기간
```

**공백 정리와 중복 제거**　같은 문구는 한 번만 검색합니다. 총 질의 상한은 3개입니다.

```
책을 빌릴 수 있는 기간은?
대출 기간
```

**같은 인덱스에 각각 질의**　각 검색의 후보 상한은 2개입니다. 실패 시 부분 결과를 성공으로 넘기지 않습니다.

```
셀 04 · 대출 기간은 며칠인가요?
두 번째 검색어: 대출 기간
```

**청크 ID 중복 제거와 절단**　처음 나온 순서를 보존해 합친 뒤 최종 1개를 전달합니다.

```
[library-1:0] + [library-1:0]
중복 제거 → [library-1:0]
```
